# Calculo de D = C^T d C, S = D # G e s = A^T S A

Este notebook calcula as etapas da transformacao uma saida por vez.

Primeiro calcula-se `D = C^T d C`. Para cada coluna `j` de `C`:

1. Calcula-se a coluna intermediaria `D'[:, j] = d C[:, j]`, multiplicando cada linha de `d` pela coluna `j` de `C`.
2. Essa coluna intermediaria e entao multiplicada pelas linhas de `C^T`, gerando uma coluna de `D`.

Depois calcula-se `S = D # G`, onde `#` e multiplicacao ponto a ponto.

Por fim calcula-se `s = A^T S A`, repetindo o mesmo processo coluna por coluna.

In [ ]:
import numpy as np
from fractions import Fraction

C = np.array([
    [-1, 0,  0,  0],
    [ 0, 1, -1, -1],
    [ 1, 1,  1,  0],
    [ 0, 0,  0,  1],
], dtype=int)

d = np.array([
    [ 0,  1,  2,  3],
    [ 4,  5,  6,  7],
    [ 8,  9, 10, 11],
    [12, 13, 14, 15],
], dtype=int)

print("C =")
print(C)
print("\nd =")
print(d)
print("\nC^T =")
print(C.T)


In [ ]:
def dot_expression(a, b):
    return " + ".join(f"({x})*({y})" for x, y in zip(a, b))


def print_column_step(j, intermediate_col, output_col):
    print(f"Coluna {j} de C:")
    print(C[:, j])
    print(f"\n1) Calculando D'[:, {j}] = d C[:, {j}]")

    for row in range(d.shape[0]):
        expr = dot_expression(d[row, :], C[:, j])
        print(f"D'[{row}, {j}] = {expr} = {intermediate_col[row]}")

    print(f"\nD'[:, {j}] = {intermediate_col.tolist()}")
    print(f"\n2) Calculando D[:, {j}] = C^T D'[:, {j}]")

    for row in range(C.T.shape[0]):
        expr = dot_expression(C.T[row, :], intermediate_col)
        print(f"D[{row}, {j}] = {expr} = {output_col[row]}")

    print(f"\nD[:, {j}] = {output_col.tolist()}")


## Calculo de D = C^T d C

A matriz `D' = d C` nao precisa ser formada inteira de uma vez. Abaixo, cada coluna de `D'` e calculada separadamente e imediatamente usada para produzir a coluna correspondente de `D`.

In [ ]:
D = np.zeros((4, 4), dtype=int)
intermediate_columns = []

for j in range(C.shape[1]):
    intermediate_col = d @ C[:, j]
    output_col = C.T @ intermediate_col

    D[:, j] = output_col
    intermediate_columns.append(intermediate_col)

    print("=" * 70)
    print_column_step(j, intermediate_col, output_col)
    print()


## Resultado de D

In [ ]:
D_prime = np.column_stack(intermediate_columns)
D_direct = C.T @ d @ C

print("D' = d C =")
print(D_prime)
print("\nD = C^T D' = C^T d C =")
print(D)

assert np.array_equal(D, D_direct)
print("\nVerificacao: D e igual a C^T @ d @ C.")


## Multiplicacao ponto a ponto: S = D # G

O operador `#` representa multiplicacao elemento a elemento. Assim, cada valor de `S` e calculado como `S[i, j] = D[i, j] * G[i, j]`.

In [ ]:
G = np.array([
    [Fraction(0),    Fraction(-3, 2), Fraction(-1, 2), Fraction(-2)],
    [Fraction(-9, 2), Fraction(9),     Fraction(3),     Fraction(15, 2)],
    [Fraction(-3, 2), Fraction(3),     Fraction(1),     Fraction(5, 2)],
    [Fraction(-6),    Fraction(21, 2), Fraction(7, 2),  Fraction(8)],
], dtype=object)

S = D.astype(object) * G

print("G =")
print(G)
print("\nS = D # G =")

for i in range(D.shape[0]):
    for j in range(D.shape[1]):
        print(f"S[{i}, {j}] = D[{i}, {j}] * G[{i}, {j}] = ({D[i, j]})*({G[i, j]}) = {S[i, j]}")

print("\nS =")
print(S)


## Calculo de s = A^T S A

Agora o mesmo processo e repetido para `s = A^T S A`, calculando primeiro `S' = A^T S`. Para cada linha `i` de `A^T`:

1. Calcula-se a linha intermediaria `S'[i, :] = A^T[i, :] S`, multiplicando uma linha de `A^T` por cada coluna de `S`.
2. Essa linha intermediaria e multiplicada pelas colunas de `A`, gerando a linha correspondente de `s`.

In [ ]:
A = np.array([
    [1,  0],
    [1,  1],
    [1, -1],
    [0,  1],
], dtype=object)

print("A =")
print(A)
print("\nA^T =")
print(A.T)


In [ ]:
def print_s_row_step(row, intermediate_row, output_row):
    print(f"Linha {row} de A^T:")
    print(A.T[row, :])
    print(f"\n1) Calculando S'[{row}, :] = A^T[{row}, :] S")

    for col in range(S.shape[1]):
        expr = dot_expression(A.T[row, :], S[:, col])
        print(f"S'[{row}, {col}] = {expr} = {intermediate_row[col]}")

    print(f"\nS'[{row}, :] = {intermediate_row.tolist()}")
    print(f"\n2) Calculando s[{row}, :] = S'[{row}, :] A")

    for col in range(A.shape[1]):
        expr = dot_expression(intermediate_row, A[:, col])
        print(f"s[{row}, {col}] = {expr} = {output_row[col]}")

    print(f"\ns[{row}, :] = {output_row.tolist()}")


In [ ]:
s = np.zeros((2, 2), dtype=object)
s_intermediate_rows = []

for row in range(A.T.shape[0]):
    intermediate_row = A.T[row, :] @ S
    output_row = intermediate_row @ A

    s[row, :] = output_row
    s_intermediate_rows.append(intermediate_row)

    print("=" * 70)
    print_s_row_step(row, intermediate_row, output_row)
    print()


## Resultado final de s

In [ ]:
S_prime = np.vstack(s_intermediate_rows)
s_direct = A.T @ S @ A

print("S' = A^T S =")
print(S_prime)
print("\ns = S' A = A^T S A =")
print(s)

assert np.array_equal(s, s_direct)
print("\nVerificacao: s e igual a A.T @ S @ A.")


## Versao em fluxo com acumulador em s

Nesta versao, cada linha de `S` e calculada uma unica vez e usada imediatamente para gerar a linha correspondente de `S' = S A`.

Para cada linha `r`:

1. Calcula-se uma linha `D'[r, :] = C^T[r, :] d`.
2. Calcula-se uma linha `D[r, :] = D'[r, :] C`.
3. Calcula-se uma linha `S[r, :] = D[r, :] # G[r, :]`.
4. Calcula-se a linha `S'[r, :] = S[r, :] A`.
5. Acumula-se `A[r, out_row] * S'[r, :]` em `s[out_row, :]`.

In [ ]:
def compute_D_row(row):
    c_t_row = C.T[row, :]
    D_prime_row = c_t_row @ d
    d_row = D_prime_row @ C

    print(f"Linha {row} de C^T:")
    print(c_t_row)
    print(f"\n1) Calculando linha intermediaria D'[{row}, :] = C^T[{row}, :] d")

    for col in range(d.shape[1]):
        expr = dot_expression(c_t_row, d[:, col])
        print(f"D'[{row}, {col}] = {expr} = {D_prime_row[col]}")

    print(f"\nD'[{row}, :] = {D_prime_row.tolist()}")
    print(f"\n2) Calculando D[{row}, :] = D'[{row}, :] C")

    for col in range(C.shape[1]):
        expr = dot_expression(D_prime_row, C[:, col])
        print(f"D[{row}, {col}] = {expr} = {d_row[col]}")

    print(f"\nD[{row}, :] = {d_row.tolist()}")
    return d_row


def compute_S_row(row):
    d_row = compute_D_row(row)
    s_row = np.zeros(4, dtype=object)

    print(f"\n3) Calculando S[{row}, :] = D[{row}, :] # G[{row}, :]")
    for col in range(d_row.shape[0]):
        s_row[col] = d_row[col] * G[row, col]
        print(f"S[{row}, {col}] = D[{row}, {col}] * G[{row}, {col}] = ({d_row[col]})*({G[row, col]}) = {s_row[col]}")

    print(f"\nS[{row}, :] = {s_row.tolist()}")
    return s_row


def accumulate_s_from_S_row(row, s_row, s_accum):
    S_prime_row = s_row @ A

    print(f"\n4) Calculando linha intermediaria S'[{row}, :] = S[{row}, :] A")
    for col in range(A.shape[1]):
        expr = dot_expression(s_row, A[:, col])
        print(f"S'[{row}, {col}] = {expr} = {S_prime_row[col]}")

    print(f"\n5) Acumulando A[{row}, out_row] * S'[{row}, :] em s[out_row, :]")
    for out_row in range(A.shape[1]):
        weight = A[row, out_row]
        contribution = weight * S_prime_row
        s_accum[out_row, :] += contribution
        print(f"s[{out_row}, :] += A[{row}, {out_row}] * S'[{row}, :] = ({weight}) * {S_prime_row.tolist()} = {contribution.tolist()}")
        print(f"s[{out_row}, :] parcial = {s_accum[out_row, :].tolist()}")

    return S_prime_row


In [ ]:
s_stream = np.zeros((2, 2), dtype=object)
S_prime_rows = []

for row in range(C.shape[0]):
    print("=" * 70)
    print(f"Fluxo da linha {row}")

    S_row = compute_S_row(row)
    S_prime_row = accumulate_s_from_S_row(row, S_row, s_stream)
    S_prime_rows.append(S_prime_row)

    print(f"\ns parcial apos linha {row}:")
    print(s_stream)
    print()


## Resultado da versao com acumulador em s

In [ ]:
S_prime_stream = np.vstack(S_prime_rows)

print("S_prime_stream = S A =")
print(S_prime_stream)
print("\ns_stream = A^T S_prime_stream = A^T S A =")
print(s_stream)

assert np.array_equal(S_prime_stream, S @ A)
assert np.array_equal(s_stream, s)
print("\nVerificacao: a versao com acumulador em s gera os mesmos S' e s.")
